# SRQ-FLY Priority 2C: implicit Ridge initialization
Final synthetic system gate before real-data train-only equivalence. No dataset or held-out test is used. Run on a Tesla T4 from top to bottom.

In [ ]:
# Edit only these repository/path values.
REPO_GIT_URL='https://github.com/ZaPhat206/SOHO-CL.git'
REPO_BRANCH='experiment/soho-selfcontained'
WORK_DIR='/content/SOHO-CL'
OUTPUT_DIR='/content/srq_priority2c_output'

In [ ]:
# Fresh clone, dependencies and T4 check.
import hashlib,json,os,shutil,subprocess,sys,time
from pathlib import Path
os.chdir('/content')
repo=Path(WORK_DIR)
if repo.exists(): shutil.rmtree(repo)
subprocess.run(['git','clone','--branch',REPO_BRANCH,'--single-branch',REPO_GIT_URL,WORK_DIR],check=True)
os.chdir(WORK_DIR)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt'],check=True)
import torch
assert torch.cuda.is_available(),'Select a GPU runtime'
print('repo commit:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())
print('GPU:',torch.cuda.get_device_name(0))

In [ ]:
# Immutable protocol/source check. Do not edit these hashes.
CONFIG='configs/srq_fly_priority2c_implicit_ridge_memory.json'
RUNNER='tools/srq_fly_priority2c_memory_benchmark.py'
EXPECTED_CONFIG_SHA256='68ba2cb904ad065c91e778e0532f871e2448ec126e8a431a85f6bafd681f7e83'
EXPECTED_RUNNER_SHA256='c220368cc55e660cf0a30a7bedd1d59aa2ec86aa41fe0904caf27edb0aab61b3'
EXPECTED_LEARNER_SHA256='40edac2e2cc88faac549f5c87217f3143d815bf53ecad8a37dfdb22c112691ae'
EXPECTED_SYSTEM_WORKER_SHA256='85e2d7f8a27f8081148a88bf88dcd374af20aa39f2a77691c9ca744a2ff0d96c'
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
assert sha(CONFIG)==EXPECTED_CONFIG_SHA256
assert sha(RUNNER)==EXPECTED_RUNNER_SHA256
assert sha('methods/srq_fly_optimized/learner.py')==EXPECTED_LEARNER_SHA256
assert sha('tools/srq_fly_system_benchmark.py')==EXPECTED_SYSTEM_WORKER_SHA256
assert not subprocess.check_output(['git','status','--porcelain'],text=True).strip(),'Repository must be clean'
print('LOCKED SOURCE CHECK: PASS')

In [ ]:
# CPU/synthetic correctness gate.
subprocess.run([sys.executable,'-m','pytest','-q','tests/test_srq_fly_optimized.py','tests/test_srq_fly_priority2b_memory.py','tests/test_srq_fly_priority2c_memory.py'],check=True)
print('PRIORITY-2C CORRECTNESS GATE: PASS')

In [ ]:
# Locked T4 benchmark: 24 isolated workers, resumable within this runtime.
Path(OUTPUT_DIR).mkdir(parents=True,exist_ok=True)
command=[sys.executable,'-u',RUNNER,'--config',CONFIG,'--output-dir',OUTPUT_DIR,'--device','cuda','--require-clean-git']
print('PRIORITY-2C START: 3 methods x (1 warm-up + 7 measured).',flush=True)
completed=subprocess.run(command)
assert completed.returncode==0,'Priority-2C failed; return the complete traceback without editing gates.'
RESULT=Path(OUTPUT_DIR)/'priority2c_memory_results.json'
assert RESULT.is_file()
payload=json.loads(RESULT.read_text())
print('PRIORITY-2C DECISION:',payload['status'])

In [ ]:
# Compact result table.
import pandas as pd
rows=[]
for row in payload['summaries']:
    rows.append({'method':row['label'],'update_s_median':row['update_seconds']['median'],'peak_allocated_GiB':row['peak_allocated_bytes']['median']/2**30,'peak_reserved_GiB':row['peak_reserved_bytes']['median']/2**30,'state_MiB':row['persistent_state_bytes']/2**20,'solver_residual_max':row['maximum_solver_relative_residual']})
display(pd.DataFrame(rows))
print(json.dumps({'status':payload['status'],'paired_update_ratio':payload['paired_update_ratio_to_priority2b'],'paired_peak_ratio':payload['paired_peak_ratio_to_priority2b'],'maximum_relative_logit_drift':payload['maximum_relative_logit_drift'],'gates':payload['gates']},indent=2))
print('STOP here and return the ZIP. This notebook does not authorize test evaluation.')

In [ ]:
# Export audit evidence.
bundle=Path('/content/srq_fly_priority2c_memory')
if bundle.exists(): shutil.rmtree(bundle)
bundle.mkdir()
shutil.copy2(CONFIG,bundle/Path(CONFIG).name)
shutil.copy2(RESULT,bundle/RESULT.name)
shutil.copytree(OUTPUT_DIR,bundle/'workers')
(bundle/'repo_commit.txt').write_text(subprocess.check_output(['git','rev-parse','HEAD'],text=True))
archive=shutil.make_archive('/content/srq_fly_priority2c_memory','zip',bundle.parent,bundle.name)
print('artifact:',archive,'sha256:',sha(archive))
from google.colab import files
files.download(archive)